In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Train & Save Base Models

In [ ]:
import joblib
from sklearn.metrics import accuracy_score, classification_report

# Load feature matrices and splits
X_train_final, X_test_final = joblib.load('/kaggle/working/feature_matrices.pkl')
_, _, y_train, y_test = joblib.load('/kaggle/working/data_splits.pkl')

print(f"Train features: {X_train_final.shape}")
print(f"Test features:  {X_test_final.shape}")
print(f"Train labels: {len(y_train)}")
print(f"Test labels: {len(y_test)}")

# 1. Train Logistic Regression

In [ ]:
#Train Logistic Regression (fast baseline)

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_final, y_train)

y_pred_lr = lr.predict(X_test_final)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.2f}%)")
print(classification_report(y_test, y_pred_lr, target_names=['Fake', 'True']))

joblib.dump(lr, '/kaggle/working/model_lr.pkl')
print("Saved: model_lr.pkl")

# 2. Train XGBoost

In [ ]:
#Train XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_final, y_train)

y_pred_xgb = xgb.predict(X_test_final)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"XGBoost Accuracy: {acc_xgb:.4f} ({acc_xgb*100:.2f}%)")
print(classification_report(y_test, y_pred_xgb, target_names=['Fake', 'True']))

joblib.dump(xgb, '/kaggle/working/model_xgb.pkl')
print("Saved: model_xgb.pkl")

# 3. Train LightGBM

In [ ]:
#Train LightGBM

from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train_final, y_train)

y_pred_lgbm = lgbm.predict(X_test_final)
acc_lgbm = accuracy_score(y_test, y_pred_lgbm)

print(f"LightGBM Accuracy: {acc_lgbm:.4f} ({acc_lgbm*100:.2f}%)")
print(classification_report(y_test, y_pred_lgbm, target_names=['Fake', 'True']))

joblib.dump(lgbm, '/kaggle/working/model_lgbm.pkl')
print("Saved: model_lgbm.pkl")

In [ ]:
# After training lr, xgb, lgbm, and stacking meta-model:
y_pred_lr = lr.predict(X_test_final)
acc_lr = accuracy_score(y_test, y_pred_lr)

y_pred_xgb = xgb.predict(X_test_final)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

y_pred_lgbm = lgbm.predict(X_test_final)
acc_lgbm = accuracy_score(y_test, y_pred_lgbm)

# EVALUATION MATRIX

In [ ]:
print("\n=== BASE MODEL RESULTS ===")
print(f"{'Model / Framework':35} {'Accuracy (%)'}")
print("-" * 50)
print(f"{'Logistic Regression':35} {acc_lr*100:.4f}%")  
print(f"{'XGBoost':35} {acc_xgb*100:.4f}%")
print(f"{'LightGBM':35} {acc_lgbm*100:.4f}%")
print(f"{'Baseline Deep Learning Framework':35} 99.13%")